# Notebook 12: Cumulative Citations
Does sharing buy an **early** citation advantage in this corpus, and does it survive adjustment?
<br><br>
The question comes from [Colavizza et al. (2020)](https://doi.org/10.1371/journal.pone.0230416), who analysed 367,836 PLOS and BMC articles and found that a data availability statement containing a link to data in a repository was associated with about **25% more citations** (25.36% +/- 1.07%) in a **three-year window from publication**. Two details of their design matter before anything here is compared with it. Their outcome is the *cumulative* count inside one fixed window, not a year-by-year series. And the window is anchored on the publication *month*, not the calendar year (an article published in June 2015 is counted to June 2018), with the month entered as a covariate, so the publication year itself falls inside the window.
<br><br>
Neither detail can be reproduced here. OpenAlex serves `counts_by_year`, so the calendar year is the finest resolution available, and year 0 is then a partial year whose length varies by up to twelve months between articles, which is why notebook 10 excludes it. This section therefore does not try to match their estimand or their number. It asks the two questions their result raises, on the two quantities this corpus does support:
1. **The initial gap:** do sharing articles start ahead, in their first full year?
2. **Early exposure:** are they ahead over the first three full years taken together?

Each is asked twice, in the same two ways: **unadjusted**, as a univariate comparison of the raw counts, and **adjusted**, as a contrast derived from notebook 10's model, which controls for venue impact, preprint status, open access, U.S. authorship, author count and cohort. Nothing here is reported in the manuscript. The point is to see whether an early advantage of the size Colavizza et al. report is present in these data once the covariates are in, and, if it is not, how much of one the data can still rule out.

In [1]:
from dataclasses import dataclass
from typing import Sequence

import numpy as np
import pandas as pd
from scipy import stats

from helpers import dataset
from helpers.dynamics import (
    PRIMARY_FAMILY, PRIMARY_OUTCOME, ModelSpec, TimeScale, add_time_terms, build_dynamics_frame, total_random_variance,
)
from helpers.equivalence import ALPHA, LOG_BOUND, tost_from_estimate
from helpers.r_models import GlmmResult, fit_glmm
from helpers.stats import compare_continuous

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Helper Functions
Shared by the two subsections below, so that the initial gap and the three-year window are asked in exactly the same way: one univariate test and one model-based contrast each.
<br><br>
`TypicalArticle` evaluates notebook 10's winner for an article with every covariate at its mean and the random effects at zero. The same article stands in both groups, so a difference between them is the sharing terms alone. It offers the contrast on two scales: `log_gap`, the difference on the model's own `log(1 + y)` scale, which is where the model is linear and the standard error exact; and `cumulative_contrast`, the difference in predicted citations summed over a set of years, which is interpretable in citations but non-linear in the coefficients, so its standard error is a delta-method approximation.

In [3]:
@dataclass(frozen=True)
class TypicalArticle:
    """Notebook 10's winning model evaluated for one typical article: every covariate at its mean over the
    article-years, and the random effects at zero. The group difference is then the sharing terms alone."""
    fit: GlmmResult
    scale: TimeScale
    means: dict[str, float]

    @property
    def terms(self) -> list[str]:
        return list(self.fit.fixed_effects.index)

    def design_row(self, sharing: int, k: int) -> np.ndarray:
        """One row of the fixed-effects design: this article, in that group, in year `k`."""
        k_z = (k - self.scale.k_mean) / self.scale.k_sd
        values = {
            "(Intercept)": 1.0, "is_sharing_data": float(sharing), "k_z": k_z,
            "log_k_z": (np.log(k) - self.scale.log_k_mean) / self.scale.log_k_sd,
            "is_sharing_data:k_z": float(sharing) * k_z, **self.means,
        }
        missing = set(self.terms) - set(values)
        assert not missing, f"the design is missing {sorted(missing)}"
        return np.array([values[term] for term in self.terms])

    def log_gap(self, k: int) -> dict[str, float]:
        """Sharing minus non-sharing in year `k`, on the model's own log(1 + y) scale, where the contrast is
        linear in the coefficients and its standard error is exact. The shared `log_k_z` term carries no
        interaction, so it is common to both groups and cancels; `exp(estimate)` is the ratio of the groups'
        geometric means of 1 + citations."""
        contrast = self.design_row(1, k) - self.design_row(0, k)
        estimate = float(contrast @ self.fit.fixed_effects["estimate"].to_numpy())
        se = float(np.sqrt(contrast @ self.fit.fixed_effects_cov.to_numpy() @ contrast))
        margin = stats.norm.ppf(0.975) * se
        return {
            "k": k, "gap (log)": estimate, "SE": se, "CI low": estimate - margin, "CI high": estimate + margin,
            "ratio": np.exp(estimate), "% diff": 100 * np.expm1(estimate), "p": 2 * stats.norm.sf(abs(estimate / se)),
        }

    def _half_variance(self, k: int) -> float:
        """Half the model's total random variance at `k`, the lognormal correction's term. It has no group
        term, so it is the same for both groups in a given year."""
        k_z = pd.Series([(k - self.scale.k_mean) / self.scale.k_sd])
        return float(total_random_variance(self.fit.variance_components, "k_z", k_z).iloc[0]) / 2

    def predicted_citations(self, sharing: int, k: int) -> float:
        """Predicted mean citations in year `k`, back-transformed from log(1 + y) as exp(mu + V / 2) - 1, the
        same lognormal correction notebook 10's figure uses."""
        return float(np.exp(self.design_row(sharing, k) @ self.fit.fixed_effects["estimate"].to_numpy()
                            + self._half_variance(k)) - 1)

    def cumulative_contrast(self, years: Sequence[int]) -> dict[str, float]:
        """Predicted citations summed over `years`, per group, and the difference between the groups.

        The sum of back-transformed predictions is non-linear in the coefficients, so the standard error is the
        delta method on the fixed-effects covariance, holding the variance components fixed. That is the usual
        approximation and it makes the interval slightly optimistic.
        """
        beta = self.fit.fixed_effects["estimate"].to_numpy()
        totals, gradient = {}, np.zeros(len(self.terms))
        for sharing in (0, 1):
            total, group_gradient = 0.0, np.zeros(len(self.terms))
            for k in years:
                row = self.design_row(sharing, k)
                predicted = float(np.exp(row @ beta + self._half_variance(k)) - 1)
                total += predicted
                group_gradient += (predicted + 1) * row      # d/dbeta of exp(mu + V / 2)
            totals[sharing] = total
            gradient += group_gradient if sharing else -group_gradient
        difference = totals[1] - totals[0]
        se = float(np.sqrt(gradient @ self.fit.fixed_effects_cov.to_numpy() @ gradient))
        margin = stats.norm.ppf(0.975) * se
        return {
            "non-sharing": totals[0], "sharing": totals[1], "difference": difference, "SE": se,
            "CI low": difference - margin, "CI high": difference + margin, "ratio": totals[1] / totals[0],
            "% diff": 100 * (totals[1] / totals[0] - 1), "p": 2 * stats.norm.sf(abs(difference / se)),
        }


def univariate_test(values: pd.Series, is_sharing: pd.Series) -> pd.DataFrame:
    """The project's univariate comparison of `values` between the two sharing groups, through
    `helpers.stats.compare_continuous`: it picks a t-test or Mann-Whitney by a normality check and reads every
    effect size as sharing relative to non-sharing. The one-sided test is the project's convention for the
    sharing hypothesis; the two-sided one is shown beside it because this section is exploratory."""
    frame = pd.DataFrame({"Is Sharing Data": is_sharing.astype(int), "citations": values})
    rows = {}
    for alternative in ("greater", "two-sided"):
        result, _ = compare_continuous(frame, "Is Sharing Data", "citations", alternative=alternative)
        rows[alternative] = {
            "test": result["test"], "statistic": result["statistic"], "p": result["p_val"],
            **result["effect_sizes"], "significant": result["p_val"] < ALPHA,
        }
    return pd.DataFrame(rows).T.rename_axis("alternative")


def group_summary(values: pd.Series, is_sharing: pd.Series) -> pd.DataFrame:
    """Mean, median and spread of `values` per group, plus the geometric mean of 1 + y, which is the scale the
    model works on."""
    rows = {}
    for sharing, label in ((0, "non-sharing"), (1, "sharing")):
        group = values[is_sharing == sharing]
        rows[label] = {
            "n": len(group), "mean": group.mean(), "median": group.median(), "SD": group.std(),
            "min": group.min(), "max": group.max(), "geometric mean (1+y)": np.expm1(np.log1p(group).mean()),
        }
    return pd.DataFrame(rows).T

## Data Preparation
The article-year frame of notebook 10, its time scale, and its winning `g(k)` refitted here so this notebook runs standalone. The model is `log(1 + citations) ~ is_sharing_data * k_z + log_k_z + covariates + (1 | venue) + (1 + k_z | venue:article)`, fitted by maximum likelihood with the maximal random structure, exactly as there; the fit is checked against notebook 10's recorded coefficients below.
<br><br>
The raw counts come from the same frame, pivoted to one row per article, so the univariate and the model-based comparisons are built from identical data.

In [4]:
EARLY_YEARS = [1, 2, 3]
CANDIDATE = "k + log(k)"        # notebook 10's winning g(k), carried over rather than re-selected

DYNAMICS_SPEC = ModelSpec(      # notebook 10's specification
    group="is_sharing_data",
    candidates={CANDIDATE: dict(time="{group} * k_z + log_k_z", no_interaction="{group} + k_z + log_k_z", slope="k_z")},
    covariates="has_us_author + is_open_access + has_preprint + log_number_of_authors + venue_impact + log_weeks_since_pub",
    random_effects={"full": "(1 | venue_name) + (1 + {slope} | venue_name:article)", "intercept": "(1 | venue_name/article)"},
)

DYNAMICS_DF = build_dynamics_frame(combined, FEATURES_DF)
MODEL_DATA, TIME_SCALE = add_time_terms(DYNAMICS_DF)
winner = fit_glmm(MODEL_DATA, DYNAMICS_SPEC.formula(PRIMARY_OUTCOME, CANDIDATE, "full"), PRIMARY_FAMILY)
TYPICAL = TypicalArticle(winner, TIME_SCALE, {name: MODEL_DATA[name].mean() for name in DYNAMICS_SPEC.covariate_names})

# the raw counts, one row per article, from the same frame the model was fitted on
PER_YEAR = DYNAMICS_DF.pivot(index="article", columns="k", values="citations")
IS_SHARING = DYNAMICS_DF.drop_duplicates("article").set_index("article")["is_sharing_data"]

print(f"converged: {winner.fit['converged']}, singular: {winner.fit['singular']}, article-years: {winner.fit['n_obs']}")
print(f"k mean {TIME_SCALE.k_mean:.3f}, k SD {TIME_SCALE.k_sd:.3f}")
print(f"articles with a complete year {EARLY_YEARS[0]} to {EARLY_YEARS[-1]}: {PER_YEAR[EARLY_YEARS].notna().all(axis=1).sum()} of {len(PER_YEAR)}")
winner.fixed_effects.loc[["is_sharing_data", "is_sharing_data:k_z"], ["estimate", "se", "p"]]

converged: True, singular: False, article-years: 1347
k mean 3.637, k SD 2.033
articles with a complete year 1 to 3: 234 of 234


,estimate,se,p
term,,,
is_sharing_data,0.010285,0.093316,0.912333
is_sharing_data:k_z,-0.093056,0.039543,0.019715


## Initial Gap (Year 1)
The first full calendar year after publication, which every article in the sample has. Year `k` is indexed from each article's own publication, so exposure is equal across articles by construction; the calendar year it falls in is not, and that is one of the things the adjusted contrast handles through the cohort term.

In [5]:
year_one = PER_YEAR[1]

print("Year 1, raw counts:")
display(group_summary(year_one, IS_SHARING).round(3))
univariate_test(year_one, IS_SHARING)

Year 1, raw counts:


,n,mean,median,SD,min,max,geometric mean (1+y)
non-sharing,154.0,2.422,2.0,2.599,0.0,14.0,1.712
sharing,80.0,3.650,2.0,4.661,0.0,37.0,2.548


,test,statistic,p,rank-biserial,CLES,SMD,significant
alternative,,,,,,,
greater,MW_U,7482.5,0.003131,0.214692,0.607346,0.325402,True
two-sided,MW_U,7482.5,0.006262,0.214692,0.607346,0.325402,True


In [6]:
# the same comparison from the model: on its own log scale (exact SE) and as predicted counts
year_one_gap = TYPICAL.log_gap(1)
year_one_predictions = TYPICAL.cumulative_contrast([1])
year_one_tost = tost_from_estimate(
    year_one_gap["gap (log)"], year_one_gap["SE"], -LOG_BOUND, LOG_BOUND, test="TOST of the year-1 gap",
)

print("Adjusted gap in year 1, on the model's log(1 + y) scale:")
display(pd.DataFrame([year_one_gap]).round(4))
print("The same gap as predicted citations:")
display(pd.DataFrame([year_one_predictions]).round(4))
print(f"Equivalence within +-{LOG_BOUND} on the log scale:")
pd.DataFrame([year_one_tost.as_row()]).round(4)

Adjusted gap in year 1, on the model's log(1 + y) scale:


,k,gap (log),SE,CI low,CI high,ratio,% diff,p
0,1,0.131,0.0968,-0.0587,0.3206,1.1399,13.9924,0.1759


The same gap as predicted citations:


,non-sharing,sharing,difference,SE,CI low,CI high,ratio,% diff,p
0,2.7048,3.2232,0.5184,0.3939,-0.2537,1.2905,1.1917,19.1656,0.1882


Equivalence within +-0.3 on the log scale:


,test,estimate,90% CI low,90% CI high,bound,p_TOST,outcome,smallest bound
0,TOST of the year-1 gap,0.131,-0.0282,0.2901,±0.3,0.0403,equivalent,0.2901


### Reading the Year-1 Comparison
**Unadjusted, sharing articles are clearly ahead; adjusted, they are not.** The Mann-Whitney test is significant either way it is run (one-sided *p* = 0.003, two-sided *p* = 0.006), with a rank-biserial correlation of 0.21 and CLES 0.61, meaning a randomly chosen sharing article out-cites a randomly chosen non-sharing one 61% of the time. The adjusted gap is +0.131 on the log scale, a ratio of **1.14 (+14%)**, with a 95% interval from -6% to +38% and *p* = 0.18. The second table puts the same contrast in citations, 3.22 against 2.70, and its ratio is larger at 1.19: the log-scale ratio compares geometric means of 1 + y, while the count-scale one compares means that have been back-transformed and had the 1 subtracted again, which widens the ratio. They are the same null result on two scales, and the log-scale test is the one with the exact standard error.
<br><br>
Three things are worth noting before this is read as an initial boost. The **medians are identical** (2.0 in both groups) while the means are 3.65 against 2.42 and the maxima 37 against 14, so a good part of the unadjusted result lives in the upper tail; CLES is the honest one-number summary, not a median comparison. The **adjustment roughly halves the gap**, from about +31% on the raw geometric means to +14%, which is the same pattern as everywhere else in this project: sharing articles sit in higher-impact venues, more often have a preprint, and are more recent, so their first year falls in a later and more citation-rich calendar year. And the **TOST is equivalent within +-0.3** (*p* = 0.040), so while the data cannot establish a year-1 advantage, they do exclude one larger than about +35%.
<br><br>
Colavizza et al.'s ~25% sits inside the adjusted interval. These data are compatible with an effect of that size and equally compatible with none.

## Early Exposure (Cumulative Years 1+2+3)
The first three full years taken together, which is the closest this corpus comes to Colavizza et al.'s window, and which every article in the sample completes. The unadjusted comparison is on the summed raw counts; the adjusted one sums the model's predicted citations for each of the three years.

In [7]:
early_total = PER_YEAR[EARLY_YEARS].sum(axis=1)

print(f"Years {EARLY_YEARS[0]} to {EARLY_YEARS[-1]} cumulative, raw counts:")
display(group_summary(early_total, IS_SHARING).round(3))
univariate_test(early_total, IS_SHARING)

Years 1 to 3 cumulative, raw counts:


,n,mean,median,SD,min,max,geometric mean (1+y)
non-sharing,154.0,9.208,7.0,8.517,0.0,47.0,6.468
sharing,80.0,12.025,9.0,10.045,0.0,40.0,8.389


,test,statistic,p,rank-biserial,CLES,SMD,significant
alternative,,,,,,,
greater,MW_U,7174.5,0.019344,0.164692,0.582346,0.302521,True
two-sided,MW_U,7174.5,0.038688,0.164692,0.582346,0.302521,True


In [8]:
# predicted citations per year, then their sum, with the delta-method SE on the difference
per_year_predictions = pd.DataFrame({
    label: {k: TYPICAL.predicted_citations(sharing, k) for k in EARLY_YEARS}
    for sharing, label in ((0, "non-sharing"), (1, "sharing"))
}).rename_axis("k")
per_year_predictions["difference"] = per_year_predictions["sharing"] - per_year_predictions["non-sharing"]

print("Predicted citations for a typical article, by year:")
display(per_year_predictions.round(4))
print(f"Cumulative over years {EARLY_YEARS[0]} to {EARLY_YEARS[-1]}:")
pd.DataFrame([TYPICAL.cumulative_contrast(EARLY_YEARS)]).round(4)

Predicted citations for a typical article, by year:


,non-sharing,sharing,difference
k,,,
1,2.7048,3.2232,0.5184
2,3.2138,3.5885,0.3747
3,3.3753,3.5513,0.1760


Cumulative over years 1 to 3:


,non-sharing,sharing,difference,SE,CI low,CI high,ratio,% diff,p
0,9.2938,10.3629,1.0691,1.2009,-1.2846,3.4229,1.115,11.5035,0.3733


### Reading the Three-Year Comparison
**The same pattern as year 1, and weaker.** Unadjusted, the Mann-Whitney test is significant (one-sided *p* = 0.019, two-sided *p* = 0.039, CLES 0.58, means 12.0 against 9.2 and medians 9 against 7). Adjusted, a typical sharing article is predicted **10.36** citations over the three years against **9.29** for a non-sharing one, a difference of **+1.07 citations** (95% CI -1.29 to +3.42), a ratio of **1.115 (+11.5%)**, *p* = 0.37.
<br><br>
Two checks say the prediction is behaving. The predicted non-sharing total, 9.29, sits almost on the observed non-sharing mean of 9.21, so the back-transform is calibrated; the predicted sharing total, 10.36, sits well below the observed 12.03, and that gap is precisely the covariate adjustment at work. The per-year predictions also reproduce the curve notebook 10 plots.
<br><br>
Pooling the three years **dilutes** rather than strengthens the signal, which is what notebook 10's significant sharing-by-time interaction implies: the gap is largest in year 1 and shrinks each year, so a window that averages over years 1 to 3 is averaging a shrinking quantity. The three-year contrast is also less powerful than the year-1 one despite a similar ratio, because exponentiating back to the count scale inflates the standard error; that is why its *p* is larger. For the same reason there is no TOST here: the project's smallest effect of interest is +-0.3 on the log scale, and this contrast is in citations, where no bound has been argued for.
<br><br>
The ratio interval runs from roughly -14% to +37%, so Colavizza et al.'s ~25% again falls inside it. On both framings the conclusion is the same: **an early advantage of the size they report cannot be ruled out, and cannot be shown either.**

## Conclusions
**The early advantage is present unadjusted and gone once the covariates are in**, on both framings. Unadjusted, sharing articles lead in year 1 (Mann-Whitney one-sided *p* = 0.003, CLES 0.61) and over years 1 to 3 (*p* = 0.019, CLES 0.58). Adjusted, neither holds: the year-1 gap is +14% (95% CI -6% to +38%, *p* = 0.18) and the three-year difference is +1.07 citations (95% CI -1.29 to +3.42, i.e. +11.5%, *p* = 0.37). The year-1 gap is small enough to exclude an advantage beyond about 35% (TOST *p* = 0.040); the three-year contrast is on the count scale, where no bound has been argued for, so it gets no equivalence test.
<br><br>
Colavizza et al.'s ~25% sits inside both adjusted intervals, so these data neither reproduce nor exclude it. Read with the rest of the study, the pattern is the usual one: the raw early advantage is venue and cohort rather than sharing.
<br><br>
**How much of this reaches the manuscript is still open.** The analysis was not part of the plan, and both results are null, so the current intention is a single Discussion sentence beside the Colavizza citation, giving the two adjusted estimates with their intervals and pointing at this notebook, rather than a Results subsection. If it is ever written up, the adjusted numbers go with it: quoting the unadjusted year-1 test on its own would be exactly the over-reading the review objected to.